# Adaptive Signals · Hat Yai traffic workspace

This notebook restores the latest archived blue workspace from `reference/frontend_previous` in the subsequent Junction release. The embedded Hat Yai OpenStreetMap road network and SUMO control engine are preserved.

Run the first two code cells in a Colab CPU runtime. In **Live network**, select **Start simulation**. The workspace displays simulated SUMO vehicles on real road geometry. The default demand is synthetic.

© OpenStreetMap contributors · ODbL 1.0.


In [ ]:
# Cell 1 — verify bundled source/model, install pinned core libraries, test.
import os,sys,subprocess,shutil,time,json,urllib.request,zipfile,io,base64,hashlib
from pathlib import Path
if "LIVE_PROCESS" in globals() and LIVE_PROCESS.poll() is None:
    LIVE_PROCESS.terminate();LIVE_PROCESS.wait(timeout=10)
BASE=Path(os.environ.get("ADAPTIVE_SIGNALS_WORKDIR", "/content" if Path("/content").exists() else str(Path.cwd())))
PROJECT=BASE/"adaptive_signals_hatyai_blue"
PROJECT.mkdir(parents=True,exist_ok=True)
PAYLOAD_B64='...'
PAYLOAD_SHA256="70424e79b3beff0891679b18730b30a9d3a61ad1a73989fe4d6f737b9d69a2ce"
blob=base64.b64decode(PAYLOAD_B64)
if hashlib.sha256(blob).hexdigest()!=PAYLOAD_SHA256:raise RuntimeError("Bundled payload hash mismatch")
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    manifest=json.loads(z.read("PAYLOAD_MANIFEST.json"))
    for name,record in manifest.items():
        target=(PROJECT/name).resolve()
        if not target.is_relative_to(PROJECT.resolve()):raise RuntimeError("Invalid bundled path")
        content=z.read(name)
        if len(content)!=record["bytes"] or hashlib.sha256(content).hexdigest()!=record["sha256"]:raise RuntimeError("Bundled file mismatch: "+name)
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
if shutil.which("g++") is None:raise RuntimeError("Use a Linux CPU runtime with g++ C++17.")
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(PROJECT/"requirements.txt")],check=True)
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],cwd=PROJECT,check=True)
if shutil.which('node'):
    for asset in ['viewer/app.js','viewer/model.js']:
        subprocess.run(['node','--check',asset],cwd=PROJECT,check=True)
print("Source/model verified. Tests passed. No physical signals or CCTV are connected.")
print("Report:", PROJECT/"CONTROL_OPERATOR_AUDIT.md")


In [ ]:
# Cell 2 — launch the archived blue Hat Yai workspace.
COUNTS_CSV=os.environ.get("ADAPTIVE_SIGNALS_COUNTS_CSV", "")
QUEUES_CSV=os.environ.get("ADAPTIVE_SIGNALS_QUEUES_CSV", "")
PORT=int(os.environ.get("ADAPTIVE_SIGNALS_PORT","8765"))
if "LIVE_PROCESS" in globals() and LIVE_PROCESS.poll() is None:
    LIVE_PROCESS.terminate();LIVE_PROCESS.wait(timeout=10)
args=[sys.executable,"-u",str(PROJECT/"traffic_app.py"),"--port",str(PORT)]
if COUNTS_CSV:args += ["--counts-csv",str(Path(COUNTS_CSV).resolve())]
if QUEUES_CSV:args += ["--queues-csv",str(Path(QUEUES_CSV).resolve())]
LIVE_LOG=(PROJECT/"live_server.log").open("w")
LIVE_PROCESS=subprocess.Popen(args,cwd=PROJECT,stdout=LIVE_LOG,stderr=subprocess.STDOUT)
ready=False
for _ in range(150):
    if LIVE_PROCESS.poll() is not None:break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/",timeout=1) as r:ready=b"Adaptive Signals" in r.read()
        if ready:break
    except OSError:time.sleep(.1)
if not ready:raise RuntimeError("Live startup failed. "+(PROJECT/"live_server.log").read_text()[-3000:])
print("Open Live network and choose Start simulation.")
print("Default: SUMO gap-actuated reference. Hybrid E2 is experimental; compare it with the tuned E2 reference using the same traffic.")
try:
    from google.colab import output
except ImportError:print(f"Open http://127.0.0.1:{PORT}/ in a local browser.")
else:output.serve_kernel_port_as_iframe(PORT,height="1200",cache_in_notebook=False)


In [ ]:
# Cell 3 — optional exact-source holdout reproduction; no tuning.
RUN_OPERATOR_AUDIT=os.environ.get("ADAPTIVE_SIGNALS_RUN_OPERATOR_AUDIT","0")=="1"
if RUN_OPERATOR_AUDIT:
    if "LIVE_PROCESS" in globals() and LIVE_PROCESS.poll() is None:
        LIVE_PROCESS.terminate();LIVE_PROCESS.wait(timeout=10)
    destination=BASE/"operator_audit_holdout_reproduction"
    subprocess.run([sys.executable,"tools/reproduce_control_audit.py","--split","holdout","--workers","6","--destination",str(destination)],cwd=PROJECT,check=True)
    print((destination/"reference/control_audit/holdout_summary.json").read_text())
else:
    print("Optional: set ADAPTIVE_SIGNALS_RUN_OPERATOR_AUDIT=1 to reproduce the frozen 90-run holdout.")
    print("Reproduction refuses an existing destination. Use a fresh path for a second execution.")
    print((PROJECT/"CONTROL_OPERATOR_AUDIT.md").read_text())
